# Train coffee-leaf disease-classification candidates

Notebook này chứa dataset, training loop, early stopping và evaluation thật cho các classifier candidate. DVC thực thi notebook bằng Papermill; `pipeline.py select` chỉ đọc checkpoint/metrics sau khi training hoàn tất.

In [ ]:
params_path = "params.yaml"


In [ ]:
from collections import Counter
from collections.abc import Sequence
from pathlib import Path
from typing import Any
import json
import os
import time

project_root = Path.cwd().resolve()
if not (project_root / "pipeline.py").exists():
    project_root = project_root.parent
if not (project_root / "pipeline.py").exists():
    raise RuntimeError("Run this notebook from the CoffeeLeaf-AI repository")
os.chdir(project_root)

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms

from pipeline import (
    IMAGE_EXTENSIONS,
    IMAGENET_MEAN,
    IMAGENET_STD,
    ROOT,
    build_classifier,
    load_config,
    project_path,
    reset_dir,
    seed_everything,
    write_csv,
    write_json,
)

print(f"Repository: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
class LeafCropDataset:
    """Image-folder dataset with one stable class mapping for every split."""

    def __init__(self, root: Path, classes: Sequence[str], transform: Any) -> None:
        self.classes = list(classes)
        self.transform = transform
        self.samples: list[tuple[Path, int]] = []
        for class_id, class_name in enumerate(self.classes):
            class_dir = root / class_name
            if not class_dir.exists():
                raise ValueError(f"Missing class directory: {class_dir}")
            images = [
                path for path in sorted(class_dir.iterdir())
                if path.suffix.lower() in IMAGE_EXTENSIONS
            ]
            if not images:
                raise ValueError(f"Class directory is empty: {class_dir}")
            self.samples.extend((path, class_id) for path in images)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[Any, int]:
        path, target = self.samples[index]
        with Image.open(path) as source:
            image = source.convert("RGB")
        return self.transform(image), target


def classifier_transforms(image_size: int) -> tuple[Any, Any]:
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    evaluation_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_transform, evaluation_transform


def evaluate_classifier(model: Any, loader: Any, device: Any, criterion: Any = None) -> dict[str, Any]:
    model.eval()
    targets: list[int] = []
    predictions: list[int] = []
    total_loss = 0.0
    with torch.inference_mode():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            if criterion is not None:
                total_loss += float(criterion(logits, labels).item()) * labels.size(0)
            predictions.extend(logits.argmax(1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    labels_index = list(range(len(loader.dataset.classes)))
    precision, recall, f1, support = precision_recall_fscore_support(
        targets, predictions, labels=labels_index, zero_division=0
    )
    return {
        "loss": total_loss / max(1, len(targets)),
        "accuracy": float(accuracy_score(targets, predictions)),
        "precision": precision.tolist(),
        "recall": recall.tolist(),
        "f1": f1.tolist(),
        "support": support.tolist(),
        "macro_f1": float(np.mean(f1)),
        "targets": targets,
        "predictions": predictions,
    }


In [ ]:
config = load_config(params_path)
seed = int(config["seed"])
seed_everything(seed)
settings = config["classification"]
classes = list(config["data"]["classes"])
data_root = project_path(config["data"]["processed_dir"]) / "classification"
required_splits = [data_root / split for split in ("train", "val", "test")]
missing = [str(path) for path in required_splits if not path.exists()]
if missing:
    raise RuntimeError("Prepared classification data is missing. Run `dvc repro prepare`. Missing: " + ", ".join(missing))

print(json.dumps(settings, indent=2))


In [ ]:
output_dir = reset_dir(ROOT / "models" / "classifiers")
metrics_by_candidate: dict[str, dict[str, Any]] = {}
comparison_rows: list[dict[str, Any]] = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
worker_count = 0 if os.name == "nt" else int(settings["workers"])
print(f"DataLoader workers: {worker_count}")

for candidate, candidate_settings in settings["candidates"].items():
    print(f"\n=== Training classifier: {candidate} ===")
    architecture = str(candidate_settings.get("architecture", candidate))
    image_size = int(candidate_settings["image_size"])
    train_transform, eval_transform = classifier_transforms(image_size)
    train_dataset = LeafCropDataset(data_root / "train", classes, train_transform)
    val_dataset = LeafCropDataset(data_root / "val", classes, eval_transform)
    test_dataset = LeafCropDataset(data_root / "test", classes, eval_transform)
    generator = torch.Generator().manual_seed(seed)
    loader_args = {
        "batch_size": int(settings["batch_size"]),
        "num_workers": worker_count,
        "pin_memory": device.type == "cuda",
    }
    train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_args)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_args)

    model, head = build_classifier(
        architecture, len(classes), float(candidate_settings["dropout"]),
        int(candidate_settings["unfreeze_blocks"]), bool(settings["pretrained"]),
    )
    model.to(device)
    counts = Counter(target for _, target in train_dataset.samples)
    class_weights = torch.tensor([
        len(train_dataset) / (len(classes) * counts[index])
        for index in range(len(classes))
    ], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights, label_smoothing=float(settings["label_smoothing"])
    )
    head_ids = {id(parameter) for parameter in head.parameters()}
    backbone_parameters = [
        parameter for parameter in model.parameters()
        if parameter.requires_grad and id(parameter) not in head_ids
    ]
    optimizer = torch.optim.AdamW([
        {"params": backbone_parameters, "lr": float(settings["backbone_lr"])},
        {"params": list(head.parameters()), "lr": float(settings["head_lr"])},
    ], weight_decay=float(settings["weight_decay"]))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=int(settings["epochs"]), eta_min=1e-6
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    target_path = output_dir / f"{candidate}.pt"
    history: list[dict[str, Any]] = []
    best_macro_f1 = -1.0
    stale_epochs = 0

    for epoch in range(1, int(settings["epochs"]) + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(inputs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += float(loss.item()) * labels.size(0)
            train_correct += int((logits.argmax(1) == labels).sum().item())
            train_total += labels.size(0)
        scheduler.step()
        validation = evaluate_classifier(model, val_loader, device, criterion)
        epoch_row = {
            "epoch": epoch,
            "train_loss": train_loss / max(1, train_total),
            "train_accuracy": train_correct / max(1, train_total),
            "val_loss": validation["loss"],
            "val_macro_f1": validation["macro_f1"],
        }
        history.append(epoch_row)
        print(
            f"epoch={epoch:03d} train_loss={epoch_row['train_loss']:.4f} "
            f"val_loss={epoch_row['val_loss']:.4f} val_f1={epoch_row['val_macro_f1']:.4f}"
        )
        if validation["macro_f1"] > best_macro_f1 + 1e-6:
            best_macro_f1 = validation["macro_f1"]
            stale_epochs = 0
            torch.save({
                "model_name": architecture,
                "candidate_name": candidate,
                "state_dict": model.state_dict(),
                "classes": classes,
                "image_size": image_size,
                "dropout": float(candidate_settings["dropout"]),
                "unfreeze_blocks": int(candidate_settings["unfreeze_blocks"]),
            }, target_path)
        else:
            stale_epochs += 1
            if stale_epochs >= int(settings["patience"]):
                print(f"Early stopping {candidate} at epoch {epoch}")
                break

    write_csv(output_dir / f"{candidate}_history.csv", history)
    checkpoint = torch.load(target_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])
    validation_result = evaluate_classifier(model, val_loader, device, criterion)
    test_result = evaluate_classifier(model, test_loader, device, criterion)
    matrix = confusion_matrix(
        test_result["targets"], test_result["predictions"],
        labels=list(range(len(classes))),
    ).tolist()

    sample, _ = test_dataset[0]
    sample = sample.unsqueeze(0).to(device)
    model.eval()
    with torch.inference_mode():
        for _ in range(5):
            model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        started = time.perf_counter()
        for _ in range(30):
            model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - started) * 1000 / 30

    validation_per_class = {
        name: {
            "precision": float(validation_result["precision"][index]),
            "recall": float(validation_result["recall"][index]),
            "f1": float(validation_result["f1"][index]),
            "support": int(validation_result["support"][index]),
        } for index, name in enumerate(classes)
    }
    test_per_class = {
        name: {
            "precision": float(test_result["precision"][index]),
            "recall": float(test_result["recall"][index]),
            "f1": float(test_result["f1"][index]),
            "support": int(test_result["support"][index]),
        } for index, name in enumerate(classes)
    }
    write_json(output_dir / f"{candidate}_report.json", {
        "validation_per_class": validation_per_class,
        "test_per_class": test_per_class,
        "test_confusion_matrix": matrix,
    })
    values = {
        "accuracy": validation_result["accuracy"],
        "macro_f1": validation_result["macro_f1"],
        "min_class_recall": min(item["recall"] for item in validation_per_class.values()),
        "test_accuracy": test_result["accuracy"],
        "test_macro_f1": test_result["macro_f1"],
        "test_min_class_recall": min(item["recall"] for item in test_per_class.values()),
        "latency_ms": latency_ms,
        "size_mb": target_path.stat().st_size / (1024 * 1024),
        "image_size": image_size,
        "validation_per_class": validation_per_class,
        "test_per_class": test_per_class,
    }
    metrics_by_candidate[candidate] = values
    comparison_rows.append({
        "candidate": candidate,
        **{key: values[key] for key in ("macro_f1", "min_class_recall", "latency_ms", "size_mb")},
    })
    del model, head, optimizer, scheduler, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

write_json(ROOT / "metrics" / "classifiers.json", {"candidates": metrics_by_candidate})
write_csv(ROOT / "metrics" / "classifiers.csv", comparison_rows)


In [ ]:
result = json.loads((ROOT / "metrics" / "classifiers.json").read_text(encoding="utf-8"))
result
